In [1]:
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import numpy as np

import ModelFxp12 as fxp

In [2]:
class GetChannel:
    def __init__(self, channel):
      self.channel = channel

    def __call__(self, img):
        return  torch.unsqueeze(img[self.channel], 0)

# Create Transforms
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.3403, 0.3121, 0.3214),
                         (0.2724, 0.2608, 0.2669)),
    GetChannel(1)
    #transforms.Grayscale()
])


# Create Datasets
testset = torchvision.datasets.SVHN(root='SVHN-DATA', split='test', download=True, transform=transform)

# Load Datasets
test_loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 64275384/64275384 [00:16<00:00, 3953893.63it/s]


In [3]:
class Cifar10CnnModel(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) # output: 32, 16, 16
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 32, 8, 8
            nn.Dropout(p=0.3)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 32 x 4 x 4
            #nn.Dropout(p=0.5)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(26 * 4 * 4, 10)
        )

    def forward(self, xb):
        xb = self.conv1(xb)
        #print(xb.shape)
        xb = self.conv2(xb)
        #print(xb.shape)
        xb = self.conv3(xb)
        #print(xb.shape)
        xb = self.fc(xb)

        return xb

In [4]:
# Computes the average accuracy
def evaluate_fxp(model, val_loader):

    batch_accuracies = []   # Accuracies of each forwarded batch

    for images, labels in val_loader:
        outputs = model(images)                    # Generates batch predictions

        # Computes the batch accuracy
        _, preds = torch.max(torch.from_numpy(outputs), dim=1)
        accuracy = torch.tensor(torch.sum(preds == labels).item() / len(preds))

        # Append the batch accuracy
        batch_accuracies.append(accuracy)

    # Computes the epoch average accuracy
    accuracy = torch.stack(batch_accuracies).mean() # Average accuracy

    return accuracy

In [5]:
# Create the fixed point model
def model_fxp(model, shift):

    layers = [
        # conv1
        fxp.Conv(model.conv1[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv1[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv2
        fxp.Conv(model.conv2[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv2[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv3
        fxp.Conv(model.conv3[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv3[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # fc1
        fxp.Linear(model.fc[1], shift, shift)
    ]

    return fxp.ModelFxp(layers)

In [ ]:
# Load the quantized floating point model
model = Cifar10CnnModel()
model.load_state_dict(torch.load('train2_qat_sh7_9435.pt', map_location=torch.device('cpu')))


# Report (GetChannel(1))
for shift in (range(7,9)):
    model_test = model_fxp(model, shift)
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}")
    model_test.AccumulatorsWidth()
    model_test.WeightsWidth()
    print('\n')

#   shift   accuracy
#


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear
Shift: 7, accuracy: 0.942529559135437
ConvLayers[0]: min = -1503, max = 1738 MIN_WIDTH = 12.0
ConvLayers[1]: min = -5756, max = 2679 MIN_WIDTH = 14.0
ConvLayers[2]: min = -5699, max = 4997 MIN_WIDTH = 14.0
ConvLayers[3]: min = -7039, max = 6085 MIN_WIDTH = 14.0
ConvLayers[4]: min = -11380, max = 4328 MIN_WIDTH = 15.0
ConvLayers[5]: min = -3993, max = 2465 MIN_WIDTH = 13.0
FullLayers[0]: min = -5462, max = 8555 MIN_WIDTH = 15.0
ConvLayers[0] min width: 3.0
ConvLayers[1] min width: 3.0
ConvLayers[2] min width: 3.0
ConvLayers[3] min width: 3.0
ConvLayers[4] min width: 3.0
ConvLayers[5] min width: 3.0
FullLayers[0] min width: 3.0


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d


In [ ]:
# Load the quantized floating point model
model = Cifar10CnnModel()
model.load_state_dict(torch.load('train2_qat_sh6_9431.pt', map_location=torch.device('cpu')))


# Report (GetChannel(1))
for shift in (range(6,7)):
    model_test = model_fxp(model, shift)
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}")
    model_test.AccumulatorsWidth()
    model_test.WeightsWidth()
    print('\n')

#   shift   accuracy
#


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear
Shift: 6, accuracy: 0.9410488605499268
ConvLayers[0]: min = -760, max = 871 MIN_WIDTH = 11.0
ConvLayers[1]: min = -2826, max = 1324 MIN_WIDTH = 13.0
ConvLayers[2]: min = -2861, max = 2524 MIN_WIDTH = 13.0
ConvLayers[3]: min = -3597, max = 3016 MIN_WIDTH = 13.0
ConvLayers[4]: min = -5556, max = 2072 MIN_WIDTH = 14.0
ConvLayers[5]: min = -1995, max = 1305 MIN_WIDTH = 12.0
FullLayers[0]: min = -2750, max = 4235 MIN_WIDTH = 14.0
ConvLayers[0] min width: 3.0
ConvLayers[1] min width: 3.0
ConvLayers[2] min width: 3.0
ConvLayers[3] min width: 3.0
ConvLayers[4] min width: 3.0
ConvLayers[5] min width: 3.0
FullLayers[0] min width: 2.0




In [ ]:
# Load the quantized floating point model
model = Cifar10CnnModel()
model.load_state_dict(torch.load('train2_qat_sh5_9420.pt', map_location=torch.device('cpu')))


# Report (GetChannel(1))
for shift in (range(5,6)):
    model_test = model_fxp(model, shift)
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}")
    model_test.AccumulatorsWidth()
    model_test.WeightsWidth()
    print('\n')

#   shift   accuracy
#


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear
Shift: 5, accuracy: 0.9356107711791992
ConvLayers[0]: min = -375, max = 430 MIN_WIDTH = 10.0
ConvLayers[1]: min = -1396, max = 670 MIN_WIDTH = 12.0
ConvLayers[2]: min = -1429, max = 1259 MIN_WIDTH = 12.0
ConvLayers[3]: min = -1748, max = 1500 MIN_WIDTH = 12.0
ConvLayers[4]: min = -2776, max = 1034 MIN_WIDTH = 13.0
ConvLayers[5]: min = -1021, max = 606 MIN_WIDTH = 11.0
FullLayers[0]: min = -1404, max = 2100 MIN_WIDTH = 13.0
ConvLayers[0] min width: 3.0
ConvLayers[1] min width: 3.0
ConvLayers[2] min width: 2.0
ConvLayers[3] min width: 2.0
ConvLayers[4] min width: 2.0
ConvLayers[5] min width: 2.0
FullLayers[0] min width: 2.0




In [14]:
# Load the quantized floating point model
model = Cifar10CnnModel()
model.load_state_dict(torch.load('train2_qat_sh7_9435.pt', map_location=torch.device('cpu')))

model_test = model_fxp(model, 9)

layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear


In [16]:
print(model_test.ConvLayers[0].w.max())
print(model_test.ConvLayers[1].w.max())
print(model_test.ConvLayers[2].w.max())
print(model_test.ConvLayers[3].w.max())
print(model_test.ConvLayers[4].w.max())
print(model_test.ConvLayers[5].w.max())
print(model_test.FullLayers[0].w.max())

10
9
8
8
8
8
7
